# Установка зависимостей
Запустите при необходимости (если пакеты еще не установлены).

In [12]:
%pip install -q openai pandas tqdm datasets

Note: you may need to restart the kernel to use updated packages.


# Импорты и структуры данных

In [13]:
import json
import os
import random
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
from datasets import load_dataset
from openai import OpenAI
from tqdm.auto import tqdm

print('Environment ready')

Environment ready


In [14]:
@dataclass
class QueryItem:
    qid: str
    question: str
    answer: Any
    table_id: str
    supporting_rows: Optional[List[int]] = None

# Функции загрузки датасета
Совместимо с текущим форматом custom FeTaQA-like из папок rag_questions + data.

In [15]:
def table_to_dataframe(tbl: pd.DataFrame | dict | list) -> pd.DataFrame:
    if isinstance(tbl, pd.DataFrame):
        return tbl.copy()
    if isinstance(tbl, dict):
        if 'rows' in tbl and 'header' in tbl:
            return pd.DataFrame(tbl['rows'], columns=tbl['header'])
        return pd.DataFrame(tbl)
    if isinstance(tbl, list):
        return pd.DataFrame(tbl)
    raise ValueError('Unsupported table format')


def load_custom_fetaqa_like(
    base_questions: str = 'rag_questions',
    data_dir: str = 'data',
    limit: Optional[int] = 200,
    sample_random: bool = True,
    random_seed: int = 42,
) -> tuple[Dict[str, pd.DataFrame], List[QueryItem]]:
    base_questions_path = Path(base_questions)
    data_dir_path = Path(data_dir)

    tables: Dict[str, pd.DataFrame] = {}
    queries: List[QueryItem] = []

    paths = sorted(
        [p for p in base_questions_path.rglob('*.json') if not p.name.endswith('.error.json')]
    )
    total_files = len(paths)
    if limit and total_files > limit:
        if sample_random:
            rng = random.Random(random_seed)
            paths = sorted(rng.sample(paths, k=limit))
            print(f'Randomly sampled {limit} files from {total_files} using seed={random_seed}')
        else:
            paths = paths[:limit]

    print(f'Loading {len(paths)} custom dataset files...')
    for jp in paths:
        rel = jp.relative_to(base_questions_path)
        csv_path = (data_dir_path / rel).with_suffix('.csv')
        table_id = str(rel.with_suffix(''))

        if csv_path.exists() and table_id not in tables:
            try:
                tables[table_id] = pd.read_csv(csv_path, sep='|', index_col=0).reset_index(drop=True)
            except Exception:
                continue

        try:
            obj = json.loads(jp.read_text(encoding='utf-8'))
        except Exception:
            continue

        if not isinstance(obj, list):
            continue

        for i, q in enumerate(obj):
            if not isinstance(q, dict):
                continue
            question = q.get('question', '')
            answer = q.get('answer', '')
            supporting_rows = q.get('supporting_rows') if isinstance(q.get('supporting_rows'), list) else None
            qid = f'{table_id}::q::{i}'
            queries.append(
                QueryItem(
                    qid=qid,
                    question=question,
                    answer=answer,
                    table_id=table_id,
                    supporting_rows=supporting_rows,
                )
            )

    return tables, queries


def load_hf_tabular_dataset(
    dataset_name: str, split: str = 'train[:200]'
) -> tuple[Dict[str, pd.DataFrame], List[QueryItem]]:
    ds = load_dataset(dataset_name, split=split)
    tables: Dict[str, pd.DataFrame] = {}
    queries: List[QueryItem] = []

    print(f'Loading {dataset_name}...')
    for i, ex in enumerate(ds):
        table = ex.get('table') or ex.get('context') or ex.get('table_data')
        if table is None:
            continue

        try:
            df = table_to_dataframe(table)
        except Exception:
            continue

        table_id = str(ex.get('table_id') or ex.get('id') or f'{dataset_name}::{i}')
        if table_id not in tables:
            tables[table_id] = df

        question = ex.get('question') or ex.get('query') or ''
        answer = ex.get('answer') or ex.get('answers') or ''

        supporting_rows = ex.get('supporting_rows')
        if not isinstance(supporting_rows, list):
            supporting_rows = None

        qid = str(ex.get('qid') or ex.get('question_id') or f'{table_id}::q::{i}')
        queries.append(
            QueryItem(
                qid=qid,
                question=question,
                answer=answer,
                table_id=table_id,
                supporting_rows=supporting_rows,
            )
        )

    return tables, queries

# Генерация query-вариантов
Поддерживаемые методы: `none`, `dual`, `llm_generated`.

In [16]:
UNIFIED_QUERY_EXPANSION_PROMPT = """Given a large table, I want to answer a question: {query}

Please provide TWO things in one response:
1. A list of column names that might contain necessary data
2. A list of keywords that might appear in table cells (categorical, from the question)

Answer ONLY with this JSON structure:
{{
  \"columns\": [\"column1\", \"column2\"],
  \"keywords\": [\"keyword1\", \"keyword2\"]
}}

Answer in the same language as the question."""

GENERATOR_MODELS = [
    'nvidia/nemotron-3-nano-30b-a3b:free',
    'openrouter/free',
]


def normalize_question_text(text: str) -> str:
    if not text:
        return ''
    return ' '.join(str(text).strip().split())


def _is_numeric_like(value: str) -> bool:
    s = value.strip().replace(',', '')
    if not s:
        return False
    return bool(re.fullmatch(r'[+-]?\d+(\.\d+)?', s))


def _build_openrouter_client() -> Optional[OpenAI]:
    api_key = os.getenv('OPENROUTER_API_KEY', '').strip()
    if not api_key:
        return None
    return OpenAI(
        api_key=api_key,
        base_url=os.getenv('OPENROUTER_BASE_URL', 'https://openrouter.ai/api/v1'),
    )


def _openrouter_unified_expansion(
    question: str,
    llm_client: Optional[OpenAI] = None,
    llm_model: Optional[str] = None,
    timeout: int = 60,
) -> Tuple[List[str], List[str]]:
    base = normalize_question_text(question)
    if not base:
        return [], []

    client = llm_client or _build_openrouter_client()
    if client is None:
        return [], []

    model_name = llm_model or os.getenv('OPENROUTER_MODEL') or GENERATOR_MODELS[0]
    prompt = UNIFIED_QUERY_EXPANSION_PROMPT.format(query=base)

    try:
        resp = client.chat.completions.create(
            model=model_name,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0,
            timeout=timeout,
        )
        text = ''
        if getattr(resp, 'choices', None):
            msg = getattr(resp.choices[0], 'message', None)
            text = getattr(msg, 'content', '') or ''
        result = json.loads(text) if text else {}
    except Exception as e:
        print(f'[WARN] OpenRouter unified expansion failed: {e}')
        return [], []

    if not isinstance(result, dict):
        return [], []

    columns = result.get('columns', [])
    if not isinstance(columns, list):
        columns = []

    columns_out: List[str] = []
    seen_columns = set()
    for col in columns:
        v = normalize_question_text(str(col))
        if v and v not in seen_columns:
            seen_columns.add(v)
            columns_out.append(v)

    keywords = result.get('keywords', [])
    if not isinstance(keywords, list):
        keywords = []

    keywords_out: List[str] = []
    seen_keywords = set()
    base_lower = base.lower()
    for kw in keywords:
        v = normalize_question_text(str(kw))
        if v and v not in seen_keywords and v.lower() in base_lower and not _is_numeric_like(v):
            seen_keywords.add(v)
            keywords_out.append(v)

    return columns_out, keywords_out


def get_query_variants(
    question: str,
    query_expansion: str,
    llm_client: Optional[OpenAI] = None,
    llm_model: Optional[str] = None,
    timeout: int = 60,
) -> List[str]:
    base = normalize_question_text(question)
    if not base:
        return []

    variants = [base]
    stem = base.rstrip(' ?!.')

    if query_expansion == 'dual':
        variants.append(f'table lookup: {stem}')
    elif query_expansion == 'llm_generated':
        schema_columns, cell_keywords = _openrouter_unified_expansion(
            question=base,
            llm_client=llm_client,
            llm_model=llm_model,
            timeout=timeout,
        )

        if schema_columns:
            variants.append(f"schema columns: {', '.join(schema_columns)} | question: {base}")
        if cell_keywords:
            variants.append(f"cell keywords: {', '.join(cell_keywords)} | question: {base}")

    uniq: List[str] = []
    seen = set()
    for v in variants:
        vv = normalize_question_text(v)
        if vv and vv not in seen:
            seen.add(vv)
            uniq.append(vv)

    return uniq

# Сборка и сохранение файла query variants

In [17]:
def build_batched_llm_prompt(questions: List[str]) -> str:
    """Create one prompt for multiple questions, aligned with UNIFIED_QUERY_EXPANSION_PROMPT."""
    header = [
        "Given a large table, I want to answer MULTIPLE questions.",
        "",
        "For EACH question, provide TWO things:",
        "1. A list of column names that might contain necessary data",
        "2. A list of keywords that might appear in table cells (categorical, from the question)",
        "",
        "Answer ONLY with this JSON structure:",
        "{\"items\":[{\"question_original\":\"...\",\"columns\":[\"column1\",\"column2\"],\"keywords\":[\"keyword1\",\"keyword2\"]}]}",
        "",
        "Rules:",
        "- Keep the language of each question.",
        "- Do not add explanations outside JSON.",
        "- Prefer concise keywords and avoid purely numeric tokens.",
        "- Ответ должен быть на языке вопроса",
        "",
        "Questions:",
    ]

    lines = []
    for idx, q in enumerate(questions, start=1):
        lines.append(f"{idx}. {normalize_question_text(q)}")

    return "\n".join(header + lines)

def _extract_json_object_from_text(text: str) -> Dict[str, Any]:
    """Extract JSON object from raw LLM text, including fenced output."""
    raw = (text or '').strip()
    if not raw:
        return {}

    try:
        parsed = json.loads(raw)
        return parsed if isinstance(parsed, dict) else {}
    except Exception:
        pass

    fenced = re.search(r'```(?:json)?\s*(\{.*\})\s*```', raw, flags=re.DOTALL)
    if fenced:
        try:
            parsed = json.loads(fenced.group(1))
            return parsed if isinstance(parsed, dict) else {}
        except Exception:
            pass

    start = raw.find('{')
    end = raw.rfind('}')
    if start != -1 and end != -1 and end > start:
        snippet = raw[start:end + 1]
        try:
            parsed = json.loads(snippet)
            return parsed if isinstance(parsed, dict) else {}
        except Exception:
            pass

    return {}


def _normalize_string_list(values: Any) -> List[str]:
    if not isinstance(values, list):
        return []
    out: List[str] = []
    seen = set()
    for item in values:
        v = normalize_question_text(str(item))
        if v and v not in seen:
            seen.add(v)
            out.append(v)
    return out


def _variants_from_columns_keywords(
    question_original: str,
    columns: List[str],
    keywords: List[str],
) -> List[str]:
    base = normalize_question_text(question_original)
    if not base:
        return []

    variants = [base]
    if columns:
        variants.append(f"schema columns: {', '.join(columns)} | question: {base}")

    base_lower = base.lower()
    filtered_keywords = [k for k in keywords if not _is_numeric_like(k) and k.lower() in base_lower]
    if filtered_keywords:
        variants.append(f"cell keywords: {', '.join(filtered_keywords)} | question: {base}")

    uniq: List[str] = []
    seen = set()
    for v in variants:
        vv = normalize_question_text(v)
        if vv and vv not in seen:
            seen.add(vv)
            uniq.append(vv)
    return uniq


def _parse_batched_response_to_variant_map(
    response_text: str,
    batch_questions: List[str],
) -> Dict[str, List[str]]:
    payload = _extract_json_object_from_text(response_text)
    items = payload.get('items', []) if isinstance(payload, dict) else []
    if not isinstance(items, list):
        return {}
    print(f"len items={len(items)}")
    if len(items) < 10:
        print(items)
    expected_questions = [normalize_question_text(q) for q in batch_questions if normalize_question_text(q)]
    expected_set = set(expected_questions)

    out: Dict[str, List[str]] = {}
    for idx, item in enumerate(items):
        if not isinstance(item, dict):
            continue

        q_raw = normalize_question_text(str(item.get('question_original', '')))
        if q_raw in expected_set:
            target_question = q_raw
        elif idx < len(expected_questions):
            target_question = expected_questions[idx]
        else:
            continue

        columns = _normalize_string_list(item.get('columns', []))
        keywords = _normalize_string_list(item.get('keywords', []))
        variants = _variants_from_columns_keywords(target_question, columns, keywords)
        if variants:
            out[target_question] = variants

    return out


def _request_batched_query_variants(
    questions: List[str],
    llm_client: OpenAI,
    llm_model: Optional[str] = None,
    timeout: int = 60,
    max_retries: int = 2,
) -> Dict[str, List[str]]:
    if not questions:
        return {}

    model_name = llm_model or os.getenv('OPENROUTER_MODEL') or GENERATOR_MODELS[0]
    prompt = build_batched_llm_prompt(questions)

    for attempt in range(max_retries + 1):
        try:
            resp = llm_client.chat.completions.create(
                model=model_name,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0,
                timeout=timeout,
            )
            text = ''
            if getattr(resp, 'choices', None):
                msg = getattr(resp.choices[0], 'message', None)
                text = getattr(msg, 'content', '') or ''
            parsed = _parse_batched_response_to_variant_map(text, questions)
            if parsed:
                return parsed
        except Exception as e:
            print(f'[WARN] Batched request failed (attempt {attempt + 1}/{max_retries + 1}): {e}')

    return {}


def _load_existing_variant_map(filepath: Path) -> Dict[str, List[str]]:
    if not filepath.exists():
        return {}
    try:
        data = json.loads(filepath.read_text(encoding='utf-8'))
    except Exception:
        return {}

    out: Dict[str, List[str]] = {}
    for item in data.get('queries', []):
        if not isinstance(item, dict):
            continue
        q_text = item.get('question_original')
        variants = item.get('variants', [])
        if q_text and isinstance(variants, list):
            cleaned = [normalize_question_text(v) for v in variants if normalize_question_text(v)]
            if cleaned:
                out[str(q_text)] = cleaned
    return out





def generate_query_variants_batched(
    questions: List[str],
    llm_client: Optional[OpenAI] = None,
    llm_model: Optional[str] = None,
    timeout: int = 60,
    batch_size: int = 100,
    max_retries: int = 2,
    parallel_workers: int = 4,
) -> Dict[str, List[str]]:
    """Generate llm_generated variants in batches with per-question fallback."""
    if batch_size < 1:
        raise ValueError('batch_size must be >= 1')
    if parallel_workers < 1:
        raise ValueError('parallel_workers must be >= 1')

    client = llm_client or _build_openrouter_client()
    if client is None:
        return {}

    normalized_questions: List[str] = []
    seen = set()
    for q in questions:
        qq = normalize_question_text(q)
        if qq and qq not in seen:
            seen.add(qq)
            normalized_questions.append(qq)

    out: Dict[str, List[str]] = {}
    if not normalized_questions:
        return out

    effective_workers = max(4, int(parallel_workers))
    batches = [
        normalized_questions[start:start + batch_size]
        for start in range(0, len(normalized_questions), batch_size)
    ]

    def _process_batch(batch: List[str]) -> Dict[str, List[str]]:
        batch_out: Dict[str, List[str]] = {}
        batch_map = _request_batched_query_variants(
            questions=batch,
            llm_client=client,
            llm_model=llm_model,
            timeout=timeout,
            max_retries=max_retries,
        )

        for q in batch:
            variants = batch_map.get(q)
            if not variants:
                try:
                    variants = get_query_variants(
                        question=q,
                        query_expansion='llm_generated',
                        llm_client=client,
                        llm_model=llm_model,
                        timeout=timeout,
                    )
                except Exception as e:
                    print(f'[WARN] Per-question fallback failed for: {q} ({e})')
                    variants = None
            if not variants:
                variants = [q]
            batch_out[q] = variants

        return batch_out

    with ThreadPoolExecutor(max_workers=effective_workers) as executor:
        futures = [executor.submit(_process_batch, batch) for batch in batches]
        for future in tqdm(as_completed(futures), total=len(futures), desc=f'Batched llm_generated ({effective_workers} threads)'):
            out.update(future.result())

    return out


def build_and_save_query_variants(
    queries: List[QueryItem],
    dataset_name: str,
    query_expansion: str,
    output_dir: str = 'query_variants',
    llm_model: Optional[str] = None,
    timeout: int = 60,
    max_questions: Optional[int] = None,
    resume_if_exists: bool = True,
    use_batched_llm_generation: bool = True,
    llm_batch_size: int = 100,
    llm_max_retries: int = 2,
    llm_parallel_workers: int = 4,
 ) -> Path:
    if query_expansion not in {'none', 'dual', 'llm_generated'}:
        raise ValueError(f'Unsupported query_expansion: {query_expansion}')

    output_path = Path(output_dir) / f'{dataset_name}_{query_expansion}.json'
    output_path.parent.mkdir(parents=True, exist_ok=True)

    existing_map = _load_existing_variant_map(output_path) if resume_if_exists else {}

    unique_questions: List[str] = []
    seen = set()
    for q in queries:
        q_text = str(q.question) if q.question is not None else ''
        if q_text and q_text not in seen:
            seen.add(q_text)
            unique_questions.append(q_text)

    if max_questions is not None:
        unique_questions = unique_questions[:max_questions]

    llm_client = _build_openrouter_client() if query_expansion == 'llm_generated' else None

    rows: List[Dict[str, Any]] = []
    generated_count = 0
    reused_count = 0

    generated_map: Dict[str, List[str]] = {}
    if query_expansion == 'llm_generated' and use_batched_llm_generation:
        to_generate = [q for q in unique_questions if q not in existing_map]
        if llm_client is None:
            print('[WARN] OPENROUTER_API_KEY is missing. Falling back to per-question generation.')
        elif to_generate:
            generated_map = generate_query_variants_batched(
                questions=to_generate,
                llm_client=llm_client,
                llm_model=llm_model,
                timeout=timeout,
                batch_size=llm_batch_size,
                max_retries=llm_max_retries,
                parallel_workers=llm_parallel_workers,
            )
            generated_count += len(to_generate)
            print(f'[INFO] Batched generation completed for {len(to_generate)} questions')
        elif to_generate:
            print('[WARN] Batched generator is not defined yet. Run section 6.1 first.')

    for q_text in tqdm(unique_questions, desc=f'Preparing {query_expansion}'):
        if q_text in existing_map:
            variants = existing_map[q_text]
            reused_count += 1
        elif q_text in generated_map:
            variants = generated_map[q_text]
        else:
            variants = None
        # else:
        #     variants = get_query_variants(
        #         question=q_text,
        #         query_expansion=query_expansion,
        #         llm_client=llm_client,
        #         llm_model=llm_model,
        #         timeout=timeout,
        #     )
        #     generated_count += 1

        if not variants:
            variants = [q_text]

        rows.append({
            'question_original': q_text,
            'variants': variants,
        })

    payload = {
        'dataset': dataset_name,
        'query_expansion': query_expansion,
        'generator_model': llm_model or os.getenv('OPENROUTER_MODEL') or GENERATOR_MODELS[0],
        'generated_at_utc': datetime.now(timezone.utc).isoformat(),
        'total_queries': len(rows),
        'queries': rows,
    }

    output_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )

    print(f'[OK] Saved: {output_path}')
    print(f'    total={len(rows)}, reused={reused_count}, generated={generated_count}')
    return output_path

# Конфиг запуска

In [18]:
# Dataset source: custom | fetaqa | open_wikitable | tablebench
DATASET_SOURCE = 'custom'

# Query expansion method: none | dual | llm_generated
QUERY_EXPANSION = 'llm_generated'

# Common options
LIMIT = 1500
SAMPLE_RANDOM = True
RANDOM_SEED = 42
MAX_QUESTIONS = None  # set int for debug
RESUME_IF_EXISTS = True
OUTPUT_DIR = 'query_variants'

# LLM options (for llm_generated)
OPENROUTER_MODEL = os.getenv('OPENROUTER_MODEL', 'nvidia/nemotron-3-nano-30b-a3b:free')
OPENROUTER_TIMEOUT = 60

# Batched generation options (used when QUERY_EXPANSION='llm_generated')
USE_BATCHED_LLM_GENERATION = True
LLM_BATCH_SIZE = 100
LLM_MAX_RETRIES = 2
LLM_PARALLEL_WORKERS = 4

print('Config is set')

Config is set


## Оценка токенов и подбор batch size
Если модель поддерживает длинный контекст, можно отправлять несколько вопросов в одном запросе.
Ниже функции для грубой/практичной оценки токенов и выбора безопасного размера батча.

In [19]:
# Optional tokenizer backend for better token estimate
try:
    import tiktoken
except ImportError:
    tiktoken = None


def _estimate_tokens_text(text: str, model_name: Optional[str] = None) -> int:
    """Estimate token count for text.

    Priority:
    1) tiktoken (if installed),
    2) fallback heuristic ~= chars/4.
    """
    if not text:
        return 0

    if tiktoken is not None:
        # For OpenRouter models there is often no exact tokenizer mapping in tiktoken.
        # cl100k_base is a practical approximation for planning.
        try:
            enc = tiktoken.encoding_for_model(model_name) if model_name else tiktoken.get_encoding('cl100k_base')
        except Exception:
            enc = tiktoken.get_encoding('cl100k_base')
        return len(enc.encode(text))

    # Fallback heuristic when tokenizer package is unavailable
    return max(1, int(len(text) / 4))





def estimate_batch_tokens(
    questions: List[str],
    model_name: Optional[str] = None,
    expected_output_tokens_per_question: int = 80,
    static_overhead_tokens: int = 800,
    use_batched_prompt: bool = True,
) -> Dict[str, int]:
    """Estimate total token usage for one batched request.

    expected_output_tokens_per_question is a planning parameter.
    Increase it if variants are verbose.
    """
    normalized_questions = [normalize_question_text(q) for q in questions if normalize_question_text(q)]
    if use_batched_prompt:
        request_text = build_batched_llm_prompt(normalized_questions)
    else:
        # rough equivalent to per-question requests summed together
        request_text = "\n".join(normalized_questions)

    input_tokens = _estimate_tokens_text(request_text, model_name=model_name) + static_overhead_tokens
    output_tokens = len(normalized_questions) * int(expected_output_tokens_per_question)
    total_tokens = input_tokens + output_tokens

    return {
        'n_questions': len(normalized_questions),
        'input_tokens_est': int(input_tokens),
        'output_tokens_est': int(output_tokens),
        'total_tokens_est': int(total_tokens),
    }


def suggest_safe_batch_size(
    questions: List[str],
    context_limit_tokens: int = 200_000,
    reserve_for_output_tokens: int = 50_000,
    safety_ratio: float = 0.85,
    model_name: Optional[str] = None,
    expected_output_tokens_per_question: int = 80,
) -> int:
    """Find conservative max batch size under context limit.

    Available input budget = context_limit - reserve_for_output, then multiplied by safety_ratio.
    """
    clean_questions = [normalize_question_text(q) for q in questions if normalize_question_text(q)]
    if not clean_questions:
        return 0

    max_input_budget = int((context_limit_tokens - reserve_for_output_tokens) * safety_ratio)
    if max_input_budget <= 0:
        return 0

    lo, hi = 1, len(clean_questions)
    best = 1

    while lo <= hi:
        mid = (lo + hi) // 2
        est = estimate_batch_tokens(
            clean_questions[:mid],
            model_name=model_name,
            expected_output_tokens_per_question=expected_output_tokens_per_question,
            static_overhead_tokens=800,
            use_batched_prompt=True,
        )
        if est['input_tokens_est'] <= max_input_budget:
            best = mid
            lo = mid + 1
        else:
            hi = mid - 1

    return best


def print_batch_estimate_report(
    questions: List[str],
    batch_size: int = 1000,
    model_name: Optional[str] = None,
) -> None:
    subset = [q for q in questions if normalize_question_text(q)][:batch_size]
    if not subset:
        print('No questions to estimate.')
        return

    est = estimate_batch_tokens(
        subset,
        model_name=model_name,
        expected_output_tokens_per_question=80,
        static_overhead_tokens=800,
        use_batched_prompt=True,
    )
    safe_bs = suggest_safe_batch_size(
        [q for q in questions if normalize_question_text(q)],
        context_limit_tokens=200_000,
        reserve_for_output_tokens=50_000,
        safety_ratio=0.85,
        model_name=model_name,
        expected_output_tokens_per_question=80,
    )

    print('Batch estimate report')
    print(f"  questions_in_estimate: {est['n_questions']}")
    print(f"  input_tokens_est:      {est['input_tokens_est']}")
    print(f"  output_tokens_est:     {est['output_tokens_est']}")
    print(f"  total_tokens_est:      {est['total_tokens_est']}")
    print('  context_limit_tokens:  200000')
    print(f"  suggested_safe_batch:  {safe_bs}")
    print('')
    print('Note: this is an estimate. Use API usage fields for exact post-factum token counts when available.')


# Загрузка вопросов

In [ ]:
if DATASET_SOURCE == 'custom':
    _, queries = load_custom_fetaqa_like(
        limit=LIMIT,
        sample_random=SAMPLE_RANDOM,
        random_seed=RANDOM_SEED,
    )
    dataset_name = 'custom'
elif DATASET_SOURCE == 'fetaqa':
    _, queries = load_hf_tabular_dataset('DongfuJiang/FeTaQA', split='train[:250]')
    dataset_name = 'fetaqa'
elif DATASET_SOURCE == 'open_wikitable':
    _, queries = load_hf_tabular_dataset('open-wikitable', split='train[:250]')
    dataset_name = 'open_wikitable'
else:
    raise ValueError(DATASET_SOURCE)

print(f'Loaded queries: {len(queries)}')
print('Example question:')
print(queries[0].question if queries else 'No questions loaded')

Randomly sampled 1500 files from 5011 using seed=42
Loading 1500 custom dataset files...
Loaded queries: 10488
Example question:
Какие персонажи в таблице являются чистокровными?


In [10]:
# Run this after loading queries to estimate whether batch=1000 fits context
all_questions = [q.question for q in queries]
print_batch_estimate_report(
    questions=all_questions,
    batch_size=1000,
    model_name=OPENROUTER_MODEL,
 )

Batch estimate report
  questions_in_estimate: 1000
  input_tokens_est:      33427
  output_tokens_est:     80000
  total_tokens_est:      113427
  context_limit_tokens:  200000
  suggested_safe_batch:  3435

Note: this is an estimate. Use API usage fields for exact post-factum token counts when available.


# Генерация и сохранение

In [22]:
output_path = build_and_save_query_variants(
    queries=queries,
    dataset_name=dataset_name,
    query_expansion=QUERY_EXPANSION,
    output_dir=OUTPUT_DIR,
    llm_model=OPENROUTER_MODEL,
    timeout=OPENROUTER_TIMEOUT,
    max_questions=MAX_QUESTIONS,
    resume_if_exists=RESUME_IF_EXISTS,
    use_batched_llm_generation=USE_BATCHED_LLM_GENERATION,
    llm_batch_size=LLM_BATCH_SIZE,
    llm_max_retries=LLM_MAX_RETRIES,
    llm_parallel_workers=LLM_PARALLEL_WORKERS,
)
output_path

Preparing llm_generated:   0%|          | 0/10313 [00:00<?, ?it/s]

[OK] Saved: query_variants/custom_llm_generated.json
    total=10313, reused=10313, generated=0


PosixPath('query_variants/custom_llm_generated.json')

# Быстрая проверка сохраненного файла

In [ ]:
saved = json.loads(Path(output_path).read_text(encoding='utf-8'))
print('dataset:', saved.get('dataset'))
print('query_expansion:', saved.get('query_expansion'))
print('total_queries:', saved.get('total_queries'))

preview = saved.get('queries', [])[:3]
for i, row in enumerate(preview, start=1):
    print('=' * 100)
    print(f'[{i}] question_original: {row.get("question_original")}')
    print(f'    variants: {row.get("variants")}')